# Notebook 2 of 3 — Agent Definition
Owner: Hyunju Yu — AI Engineer

---

### PaySprint AI Investment Agent — Three-Notebook Workflow

| Step | Notebook | Owner | Output |
|------|----------|-------|--------|
| 1 | `data_pipeline` | Reema Eid (Data Engineer) | Validated + ranked stock universe → `data/screener_override.json`, `data/master_stock_data.csv` |
| **2** | **`agent_definition` (this notebook)** | **Hyunju Yu (AI Engineer)** | **ReAct agent traces → `data/traces/trace_*.json`** |
| 3 | `traces_evaluation` | Quang Tran (PM) | LLM judge scores, backtesting, consistency analysis |

---

### What this notebook does

This notebook loads the validated stock rankings produced by Notebook 1, then runs the
PaySprint investment agent end-to-end against those candidates.

The agent follows the ReAct pattern: it reasons about what to do, calls a tool,
observes the result, then decides what to do next — repeating until it writes the final report.

In [1]:
# === API key + provider setup ================================================
# This notebook accepts EITHER an OpenAI key OR a Google Gemini key.
# It auto-detects which one is present in your .env and routes the agent to it.
# Locally we use Gemini (free tier) via its OpenAI-compatible endpoint, so the
# same `openai` SDK + function-calling code path works for both providers.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'])

from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
import paysprint_agent as core

GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'

# --- 1. Detect which provider's key is available (Gemini preferred locally) ---
if os.getenv('GEMINI_API_KEY'):
    PROVIDER = 'gemini'
    print('Gemini API key loaded from .env file')
elif os.getenv('OPENAI_API_KEY'):
    PROVIDER = 'openai'
    print('OpenAI API key loaded from .env file')
else:
    from getpass import getpass
    choice = input('No key found in .env. Use [g]emini or [o]penai? ').strip().lower()
    if choice.startswith('o'):
        PROVIDER = 'openai'
        os.environ['OPENAI_API_KEY'] = getpass('Paste your OpenAI API key here: ')
    else:
        PROVIDER = 'gemini'
        os.environ['GEMINI_API_KEY'] = getpass('Paste your Gemini API key here: ')
    print(f'{PROVIDER.upper()} API key set for this session')

# --- 2. Route the backbone's client factory to the detected provider ----------
# core._get_client() is used by run_agent, test_rejection, and llm_judge, so this
# single override covers the whole notebook WITHOUT editing paysprint_agent.py.
_ORIG_GET_CLIENT = core._get_client

def _routed_get_client():
    if os.getenv('USE_FAKE_LLM', '0') == '1':
        return _ORIG_GET_CLIENT()                      # offline demo mode (fake LLM)
    if PROVIDER == 'gemini':
        return OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url=GEMINI_BASE_URL)
    return OpenAI(api_key=os.environ['OPENAI_API_KEY'])

core._get_client = _routed_get_client

# --- 3. Teach estimate_cost() the Gemini list prices (per 1M tokens) ----------
core.MODEL_PRICES.setdefault('gemini-2.5-flash',      {'input': 0.30, 'output': 2.50})
core.MODEL_PRICES.setdefault('gemini-2.5-flash-lite', {'input': 0.10, 'output': 0.40})

print(f'Provider routed: {PROVIDER.upper()}')


Gemini API key loaded from .env file
Provider routed: GEMINI


---
## Step 1 - Choose the LLMs

The agent uses two different LLMs:
- MODEL_REASONING: the smarter, more expensive model that runs the orchestrator and writes the final report
- MODEL_SUMMARY: the faster, cheaper model used for evaluation


In [2]:
if 'core' not in globals():
    import paysprint_agent as core
if 'PROVIDER' not in globals():
    PROVIDER = 'gemini' if os.getenv('GEMINI_API_KEY') else 'openai'

if PROVIDER == 'gemini':
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')        # main agent / report
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gemini-2.5-flash-lite')   # cheaper / evaluation
else:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gpt-4o-mini')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')

# Keep the backbone's module-level defaults in sync (llm_judge uses MODEL_SUMMARY)
core.MODEL_REASONING = MODEL_1
core.MODEL_SUMMARY   = MODEL_2

print(f'Provider:             {PROVIDER}')
print(f'Model 1 (reasoning):  {MODEL_1}')
print(f'Model 2 (comparison): {MODEL_2}')
print()
print('Pricing reference (per 1M tokens):')
for model, prices in core.MODEL_PRICES.items():
    print(f'  {model}: ${prices["input"]}/1M input, ${prices["output"]}/1M output')


Provider:             gemini
Model 1 (reasoning):  gemini-2.5-flash
Model 2 (comparison): gemini-2.5-flash-lite

Pricing reference (per 1M tokens):
  gpt-4o: $2.5/1M input, $10.0/1M output
  gpt-4o-mini: $0.15/1M input, $0.6/1M output
  gemini-2.5-flash: $0.3/1M input, $2.5/1M output
  gemini-2.5-flash-lite: $0.1/1M input, $0.4/1M output


---
## Step 1.5 — Load Pipeline Data from Notebook 1

Notebook 1 (data_pipeline) validated each candidate ticker against live market data,
scored them by trend momentum, news sentiment, and fundamentals, and exported the
ranked lists to `data/screener_override.json`.

The cell below loads that file and replaces the default `SCREENER_STOCKS` in the backbone
module — so the agent researches the highest-confidence candidates identified by the data
pipeline rather than an arbitrary static order.

In [3]:
# === Step 1.5 - Load pipeline data from Notebook 1 ==========================
# Notebook 1 scored every candidate ticker by news momentum, sentiment, and
# fundamentals, then saved the ranked lists to data/screener_override.json.
# Loading it here replaces the hardcoded SCREENER_STOCKS order so the agent
# researches the highest-ranked candidates first.
import json, os

override_path = 'data/screener_override.json'
if os.path.exists(override_path):
    with open(override_path) as f:
        pipeline_screener = json.load(f)
    core.SCREENER_STOCKS.update(pipeline_screener)
    print(f'Pipeline data loaded — SCREENER_STOCKS updated with Notebook 1 rankings:')
    for strategy, tickers in core.SCREENER_STOCKS.items():
        print(f'  {strategy}: {tickers}')
else:
    print(f'No pipeline data found at {override_path}.')
    print('Run Notebook 1 (data_pipeline) first to generate validated rankings.')
    print('Falling back to default SCREENER_STOCKS from paysprint_agent.py:')
    for strategy, tickers in core.SCREENER_STOCKS.items():
        print(f'  {strategy}: {tickers}')

Pipeline data loaded — SCREENER_STOCKS updated with Notebook 1 rankings:
  conservative: ['JNJ', 'PG', 'KO', 'VZ', 'WMT', 'MCD', 'MMM', 'ABT']
  moderate: ['HD', 'AAPL', 'MSFT', 'GOOGL', 'V', 'MA', 'AMZN', 'JPM']
  aggressive: ['NVDA', 'AMD', 'META', 'TSLA', 'PLTR', 'CRWD', 'SNOW', 'SMCI']


---
## Step 2 - Run the Agent

This is the full agent run. It will:
1. Screen stocks for the user strategy
2. Fetch technical indicators, news sentiment, and fundamentals for each stock
3. Build a purchase plan
4. Write a final investment report

In [13]:
DEMO_PROFILE = {
    'name':              'Demo Investor',
    'budget':            5000.0,
    'aggressiveness':    'moderate',
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

# Ensure core agent helpers are available when this cell is run standalone
import paysprint_agent as core
if 'run_agent' not in globals():
    from paysprint_agent import run_agent, save_trace, print_trace_summary, estimate_cost

# Provide a sensible default model if MODEL_1 is not set (provider-aware)
if 'MODEL_1' not in globals():
    MODEL_1 = getattr(core, 'MODEL_REASONING', 'gpt-4o-mini')

# --- Demo settings ----------------------------------------------------------
# Research a focused set of moderate candidates so the demo finishes well within Gemini's free-tier rate limits. 
core.SCREENER_STOCKS['moderate'] = ['AAPL', 'MSFT', 'GOOGL', 'V', 'MA']

# Give the ReAct loop enough turns to research every candidate (5 stocks x 3 tools), then call create_purchase_plan and write the final report - even if the model issues one tool call per turn.
DEMO_MAX_TURNS = 20

print('Running PaySprint agent...\n')
print(f'Profile: ${DEMO_PROFILE["budget"]:,.0f} budget | {DEMO_PROFILE["aggressiveness"]} strategy | {DEMO_PROFILE["horizon_months"]}mo horizon')
print(f'Candidates: {core.SCREENER_STOCKS["moderate"]}  |  max_turns={DEMO_MAX_TURNS}')
print('-' * 60)

try:
    result = run_agent(DEMO_PROFILE, model=MODEL_1, max_turns=DEMO_MAX_TURNS, verbose=True)
except Exception as e:
    # Convert runtime exceptions (rate limits, quota errors) into the structured error dict
    print('LLM call failed during demo run:')
    print(e)
    result = {'error': 'LLM call failed', 'error_detail': str(e)}

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
# Guard against LLM failures. run_agent now returns a structured error dict on failure
if isinstance(result, dict) and result.get('error'):
    print('LLM error during run:')
    print(result.get('error'))
    if result.get('error_detail'):
        print('\nDetails:')
        print(result.get('error_detail'))
elif not (result.get('report') or '').strip():
    print('(No report text was returned - the model ended the run without writing one.)')
else:
    print(result.get('report', ''))


Running PaySprint agent...

Profile: $5,000 budget | moderate strategy | 12mo horizon
Candidates: ['AAPL', 'MSFT', 'GOOGL', 'V', 'MA']  |  max_turns=20
------------------------------------------------------------
  [Turn 1]  screen_stocks({"aggressiveness": "moderate"}...)
             -> {"candidates": ["AAPL", "MSFT", "GOOGL", "V", "MA"], "strategy": "moderate"}...
  [Turn 2]  get_technical_indicators({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "last_price": 295.95, "slope_per_day": 0.37, "forecast_3m": 319.26, "forecast_12m...
  [Turn 2]  get_news_sentiment({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "avg_sentiment": 0.235, "article_count": 30, "top_headlines": ["Wall Street\u2019...
  [Turn 2]  get_fundamentals({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "pe_ratio": 35.8293, "eps": 8.26, "revenue_growth": 0.166, "debt_to_equity": 79.5...
  [Turn 3]  get_technical_indicators({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "last_pri

In [14]:
if isinstance(result, dict) and result.get('error'):
    print('No trace summary available due to LLM error:')
    print(result.get('error'))
    if result.get('error_detail'):
        print(result.get('error_detail'))
else:
    print_trace_summary(result)

    cost = estimate_cost(result)
    print('\nCost estimate:')
    print(f'  Model:          {cost["model"]}')
    print(f'  Input tokens:   {cost["input_tokens"]:,}')
    print(f'  Output tokens:  {cost["output_tokens"]:,}')
    print(f'  Estimated cost: ${cost["cost_usd"]:.4f} USD')


Model : gemini-2.5-flash
Turns : 8
Tokens: 15573

Tools called:
  Turn 1 -> screen_stocks
  Turn 2 -> get_technical_indicators
  Turn 2 -> get_news_sentiment
  Turn 2 -> get_fundamentals
  Turn 3 -> get_technical_indicators
  Turn 3 -> get_news_sentiment
  Turn 3 -> get_fundamentals
  Turn 4 -> get_technical_indicators
  Turn 4 -> get_news_sentiment
  Turn 4 -> get_fundamentals
  Turn 5 -> get_technical_indicators
  Turn 5 -> get_news_sentiment
  Turn 5 -> get_fundamentals
  Turn 6 -> get_technical_indicators
  Turn 6 -> get_news_sentiment
  Turn 6 -> get_fundamentals
  Turn 7 -> create_purchase_plan

Report preview (first 400 chars):
| Ticker | Allocation (USD) | Est. Shares |
|:-------|:-----------------|:------------|
| AAPL   | $1,750.00        | 5           |
| GOOGL  | $1,750.00        | 4           |
| MSFT   | $1,500.00        | 3           |

---

### RISK DISCLOSURE

Investing in the stock market involves risks, including the potential loss of principal. The value of investm

In [15]:
result['trace_id'] = 1
result['label']    = 'Demo run - moderate $5k'
result['profile']  = DEMO_PROFILE
save_trace(result, trace_id=1)
print('Saved as Trace 1')

[trace] Saved -> data\traces\trace_1.json
Saved as Trace 1


---
## Step 3 - Test Graceful Rejection

In [16]:
# test_rejection lives in paysprint_agent.py - import it so this cell is self-contained
from paysprint_agent import test_rejection

rejection_tests = [
    'What is the capital of France?',
    'Can you write me a recipe for pasta?',
]

print('Testing graceful rejection...\n')
rejection_results = []
for msg in rejection_tests:
    r = test_rejection(msg, model=MODEL_1)
    rejection_results.append(r)
    status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called)'
    print(f'Input:    "{msg}"')
    print(f'Response: {r["response"][:200]}')
    print(f'Result:   {status}')
    print()


Testing graceful rejection...

Input:    "What is the capital of France?"
Response: I am sorry, but I can only provide information and assistance related to investment research. I cannot answer general knowledge questions.
Result:   PASS (no tools called)

Input:    "Can you write me a recipe for pasta?"
Response: I'm sorry, but I'm an AI investment research assistant, and I can only help with investment-related queries. I can't provide recipes.
Result:   PASS (no tools called)




### Customize the agent instructions

In [17]:
from paysprint_agent import build_system_prompt
if 'DEMO_PROFILE' not in globals():
    DEMO_PROFILE = {'name': 'Demo Investor', 'budget': 5000, 'aggressiveness': 'moderate', 'horizon_months': 12, 'current_holdings': {}, 'preferred_sectors': ['Technology']}
print('Current system prompt (first 600 chars):')
print(build_system_prompt(DEMO_PROFILE)[:600])
print('...')

Current system prompt (first 600 chars):
You are PaySprint, a friendly and professional AI investment research assistant.
Your goal is to help everyday investors make confident, well-informed decisions.
Always write clearly - assume the user is not a finance expert.

User Profile:
  - Name:     Demo Investor
  - Budget:   $5,000.00
  - Strategy: moderate
  - Horizon:  12 months

Follow this workflow:
1. Call screen_stocks for the user strategy.
2. For each stock: call get_technical_indicators, get_news_sentiment, get_fundamentals.
3. Pick the best 3-5 stocks and explain why in plain English.
4. Call create_purchase_plan and show an e
...


In [18]:
import json as _json

def my_custom_system_prompt(profile: dict) -> str:
    return f"""You are PaySprint, a friendly and professional AI investment research assistant.
Your goal is to help everyday investors make confident, well-informed decisions.
Always write clearly - assume the user is not a finance expert.

User Profile:
  - Name:     {profile.get('name', 'Investor')}
  - Budget:   ${profile.get('budget', 0):,.2f}
  - Strategy: {profile.get('aggressiveness', 'moderate')}
  - Horizon:  {profile.get('horizon_months', 12)} months

Follow this workflow:
1. Call screen_stocks for the user strategy.
2. For each stock: call get_technical_indicators, get_news_sentiment, get_fundamentals.
3. Pick the best 3-5 stocks and explain why in plain English.
4. Call create_purchase_plan and show an easy-to-read table.
5. End with a RISK DISCLOSURE (required).

Extra requirements:
  - If a stock has negative revenue_growth, add a WARNING next to its name.
  - Rate each stock CONSERVATIVE, MODERATE, or AGGRESSIVE at the top of its summary.
  - Use bullet points for stock summaries - keep each one under 3 bullets.

If the user asks anything unrelated to investing, politely decline.
"""

core.build_system_prompt = my_custom_system_prompt
print('Custom prompt applied. Run the next cell to test it.')

Custom prompt applied. Run the next cell to test it.


In [19]:
print('Running agent with custom prompt...\n')

if 'DEMO_MAX_TURNS' not in globals():
    DEMO_MAX_TURNS = 20
custom_result = run_agent(DEMO_PROFILE, model=MODEL_1, max_turns=DEMO_MAX_TURNS, verbose=True)
if isinstance(custom_result, dict) and custom_result.get('error'):
    print('LLM error during custom prompt run:')
    print(custom_result.get('error'))
    if custom_result.get('error_detail'):
        print('\nDetails:')
        print(custom_result.get('error_detail'))
elif not (custom_result.get('report') or '').strip():
    print('(No report text was returned - the model ended the run without writing one.)')
else:
    print('\n' + '=' * 60)
    print('CUSTOM PROMPT REPORT:')
    print('=' * 60)
    print(custom_result.get('report', ''))


Running agent with custom prompt...

  [Turn 1]  screen_stocks({"aggressiveness": "moderate"}...)
             -> {"candidates": ["AAPL", "MSFT", "GOOGL", "V", "MA"], "strategy": "moderate"}...
  [Turn 2]  get_technical_indicators({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "last_price": 295.95, "slope_per_day": 0.37, "forecast_3m": 319.26, "forecast_12m...
  [Turn 2]  get_news_sentiment({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "avg_sentiment": 0.244, "article_count": 30, "top_headlines": ["Wall Street\u2019...
  [Turn 2]  get_fundamentals({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "pe_ratio": 35.8293, "eps": 8.26, "revenue_growth": 0.166, "debt_to_equity": 79.5...
  [Turn 3]  get_technical_indicators({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "last_price": 378.91, "slope_per_day": -0.4259, "forecast_3m": 352.08, "forecast_...
  [Turn 3]  get_news_sentiment({"ticker": "MSFT"}...)
             -> {"ticker": "MSFT", "avg_sentim

---
### Adjust the scoring weights

The agent scores stocks using three signals: sentiment, momentum, and news mentions.

In [20]:
# Show current scoring weights
if 'core' not in globals():
    import paysprint_agent as core
print('Current scoring weights:')
weights = getattr(core, 'SCORING_WEIGHTS', {'sentiment': 0.4, 'momentum': 0.35, 'mentions': 0.25})
for k, v in weights.items():
    print(f'  {k}: {v}')
print(f'  Total: {sum(weights.values()):.2f}  (must equal 1.0)')

Current scoring weights:
  sentiment: 0.35
  momentum: 0.45
  mentions: 0.2
  Total: 1.00  (must equal 1.0)


In [21]:
if 'core' not in globals():
    import paysprint_agent as core

core.SCORING_WEIGHTS = {
    'sentiment': 0.35,
    'momentum':  0.45,
    'mentions':  0.20,
}

total = sum(core.SCORING_WEIGHTS.values())
assert abs(total - 1.0) < 0.001, f'Weights must sum to 1.0, got {total}'
print(f'Weights updated. Total = {total:.2f}')
for k, v in core.SCORING_WEIGHTS.items():
    print(f'  {k}: {v}')

Weights updated. Total = 1.00
  sentiment: 0.35
  momentum: 0.45
  mentions: 0.2


---
## Agent behavior observations

**Agent Performance Commentary**

*Run configuration: `gemini-2.5-flash` reasoning model, moderate $5,000 / 12-month profile, candidates AAPL / MSFT / GOOGL / V / MA.*

- **Tool usage.** The agent called every tool the workflow requires: `screen_stocks` once, then `get_technical_indicators`, `get_news_sentiment`, and `get_fundamentals` for **all five** candidates, and finally `create_purchase_plan`. It completes in 8 ReAct turns and correctly chained reason, act, observe, and stopped on its own once it had enough evidence, rather than hitting the turn cap.

- **Report clarity.** The final report is written in plain English for a non-expert and clearly laid out a short per-stock rationale, a dollar/share purchase table, and the mandatory risk disclosure at the end. 

- **Strategy alignment.** For the moderate profile it chose **GOOGL, AAPL, and MSFT** leaders in large cap with low volatility and weighted them 0.40 / 0.35 / 0.25, tilting toward the strongest momentum name (GOOGL) without over concentrating. That matches a moderate risk posture.

- **Custom prompt (Task A).** My custom prompt changed the output of the report to a markdown table with risk framing and kept stock summaries to short bullets. 

- **Graceful rejection.** Off-topic prompts (like "capital of France" or "pasta recipe") were politely declined with **zero tool calls**. 

- **What I would improve with more time.** (1) News sentiment is thin. Yahoo Finance and GNews are rate-limited, so several tickers came back with only 1 to 2 articles and a neutral 0.0 score. I would add another news source and cache results. (2) The scoring weights from Task B are illustrative only. The agent currently selects stocks through LLM reasoning, not by mechanically applying those weights, so to make weight changes truly bite I would add a deterministic scoring/ranking tool the agent must call before selecting.

---

## Notebook 2 Complete | Next: Handoff to Notebook 3

**What this notebook produced:**

| File | Contents | Used by |
|------|----------|---------|
| `data/traces/trace_1.json` | Demo run: moderate $5k / 12mo | Notebook 3 (Trace 1 baseline) |
| `data/traces/trace_*.json` | All additional traces run below | Notebook 3 evaluation |

**Next step: open `traces_evaluation.ipynb` (Notebook 3)**  
Notebook 3 runs the agent across 5 diverse scenarios (conservative, aggressive, moderate, rejection tests, dual-model comparison), evaluates each report with an LLM judge, and produces the final performance analysis.